# 09 M2: จัดกลุ่มสถานีตามบทบาท (2019 vs 2023)

Pain point ของ BART: ตารางเดินรถ ความยาวขบวน และพนักงานสถานี ยังวางตามรูปแบบการเดินทางก่อน COVID ถ้าบทบาทของสถานีเปลี่ยนไปแล้ว แผนเดิมจะไม่ตรงกับการใช้งานจริง

คำถาม
1. สถานีของ BART แบ่งได้เป็นกี่แบบตามรูปแบบการใช้งาน
2. หลัง COVID สถานีไหนเปลี่ยนบทบาท และเปลี่ยนไปเป็นแบบไหน

วิธีคิด
- ใช้วันทำงาน normal ช่วง ม.ค.-เม.ย. 2019 เทียบกับ ม.ค.-เม.ย. 2023 (ช่วงเดียวกับ Q1 จึงไม่มีผลของฤดูกาลปน)
- 1 แถวต่อสถานีต่อปี ได้ 98 แถว (2019 มี 48 สถานี, 2023 มี 50 สถานี เพราะ MLPT และ BERY เปิดปี 2020)
- feature วัด "รูปทรง" ของการใช้งาน ไม่ใช่ขนาด สถานีเล็กกับสถานีใหญ่ที่ใช้งานแบบเดียวกันจึงอยู่กลุ่มเดียวกัน
  - สัดส่วน entries แต่ละชั่วโมงต่อทั้งวัน และสัดส่วน exits แต่ละชั่วโมงต่อทั้งวัน
  - อัตราส่วนผู้โดยสารเฉลี่ยต่อวันของวันหยุด/เสาร์-อาทิตย์ ต่อวันทำงาน
- StandardScaler แล้ว KMeans โดย fit ปี 2019 และ 2023 รวมกัน กลุ่มจึงมีความหมายเดียวกันทั้งสองปี และดูได้ว่าสถานีย้ายกลุ่มหรือไม่
- เลือกจำนวนกลุ่ม (k) จาก silhouette ร่วมกับความเสถียรข้าม 50 seed และตรวจกับ Hierarchical clustering (Ward)

ข้อมูลมีแค่ 98 แถว ใช้ scikit-learn บน CPU ก็พอ งานหนักอยู่ที่การสรุปจาก mart รายชั่วโมง (~5 ล้านแถว) ด้วย DuckDB

In [1]:
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

from bart import viz
from bart.config import TAB, create_views, duck, fetch_value
from bart.perf import stage
from bart.periods import AFTER, BEFORE, period_case

viz.notebook_mode()

con = duck()
create_views(con)

## 1. สร้าง feature จาก mart รายชั่วโมง

ดึงทุกชั่วโมงของวัน normal ในสองช่วงมาเก็บเป็น temp table ก่อน แล้วสรุปเป็นรูปทรงรายชั่วโมงของแต่ละสถานี
สัดส่วนคำนวณจากผลรวมทั้งช่วง (รวมทุกวันทำงานก่อนแล้วค่อยหาร) วันที่คนเยอะจึงมีน้ำหนักมากกว่าวันที่คนน้อยตามจริง

In [2]:
with stage("09.period_hours"):
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE period_hours AS
        WITH labelled AS (
            SELECT
                h.station::VARCHAR AS station,
                {period_case("h.date")} AS period,
                h.date,
                h.hour,
                h.entries,
                h.exits,
                c.is_workday
            FROM station_hour AS h
            JOIN calendar AS c ON c.date = h.date
            WHERE c.day_status = 'normal'
        )
        SELECT *
        FROM labelled
        WHERE period IS NOT NULL
    """)

    profile = con.sql("""
        SELECT
            station,
            period,
            hour,
            sum(entries) AS entries,
            sum(exits) AS exits,
            sum(entries) / sum(sum(entries)) OVER (PARTITION BY station, period) AS entry_share,
            sum(exits) / sum(sum(exits)) OVER (PARTITION BY station, period) AS exit_share
        FROM period_hours
        WHERE is_workday
        GROUP BY station, period, hour
        ORDER BY station, period, hour
    """).df()

    daily = con.sql("""
        WITH per_day AS (
            SELECT
                station,
                period,
                date,
                is_workday,
                sum(entries) AS entries,
                sum(entries + exits) AS trips_ends
            FROM period_hours
            GROUP BY station, period, date, is_workday
        )
        SELECT
            station,
            period,
            count(*) FILTER (WHERE is_workday) AS workdays,
            count(*) FILTER (WHERE NOT is_workday) AS weekend_holidays,
            avg(entries) FILTER (WHERE is_workday) AS workday_entries,
            avg(trips_ends) FILTER (WHERE NOT is_workday) / avg(trips_ends) FILTER (WHERE is_workday) AS weekend_ratio
        FROM per_day
        GROUP BY station, period
        ORDER BY station, period
    """).df()

hour_rows = fetch_value(con, "SELECT count(*) FROM period_hours")
print(f"hourly rows used: {hour_rows:,}")
print(f"station-period rows: {len(daily)}")
daily.groupby("period").agg(stations=("station", "count"), workdays=("workdays", "max"), weekend_holidays=("weekend_holidays", "max"))

[perf] 09.period_hours: 0.06s | 3.32 cores | peak RAM 0.289 GB
hourly rows used: 281,088
station-period rows: 98


,stations,workdays,weekend_holidays
period,,,
Jan-Apr 2019,48,83,36
Jan-Apr 2023,50,82,38


ภาพรวมทั้งระบบก่อนเลือกชั่วโมงที่ใช้: ชั่วโมงตี 1 ถึงตี 4 แทบไม่มีผู้โดยสาร (BART ปิดให้บริการ)
ถ้าใส่ชั่วโมงเหล่านี้ StandardScaler จะขยายค่าที่ใกล้ศูนย์ให้มีน้ำหนักเท่าชั่วโมง peak ซึ่งเป็นแค่ noise จึงใช้ 20 ชั่วโมง คือ 5:00 ถึง 0:00
สัดส่วนยังคิดจากทั้งวัน 24 ชั่วโมง

In [3]:
system_hours = con.sql("""
    SELECT
        period,
        hour,
        100 * sum(entries) / sum(sum(entries)) OVER (PARTITION BY period) AS entry_share_pct
    FROM period_hours
    WHERE is_workday
    GROUP BY period, hour
    ORDER BY period, hour
""").df()

system_hours.pivot(index="hour", columns="period", values="entry_share_pct").round(2).T

hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
period,,,,,,,,,,,,,,,,,,,,,
Jan-Apr 2019,0.39,0.07,0.00,0.0,0.05,1.11,3.67,8.12,12.38,9.51,...,3.27,4.58,7.54,12.06,10.93,5.65,2.87,2.04,1.75,1.19
Jan-Apr 2023,0.46,0.11,0.01,0.0,0.03,1.19,3.89,7.89,11.13,7.71,...,4.39,5.92,8.65,11.81,9.10,4.87,2.72,1.93,1.80,1.23


In [4]:
FEATURE_HOURS = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 0]

entry_wide = profile.pivot_table(index=["station", "period"], columns="hour", values="entry_share")
exit_wide = profile.pivot_table(index=["station", "period"], columns="hour", values="exit_share")

features = pd.DataFrame(index=entry_wide.index)
for hour in FEATURE_HOURS:
    features[f"in_{hour:02d}"] = entry_wide[hour]
for hour in FEATURE_HOURS:
    features[f"out_{hour:02d}"] = exit_wide[hour]

ratio = daily.set_index(["station", "period"])["weekend_ratio"]
features["weekend_ratio"] = ratio.loc[features.index]

feature_columns = list(features.columns)
print(f"{features.shape[0]} station-period rows x {features.shape[1]} features")
features.head()

98 station-period rows x 41 features


in_05     in_06     in_07     in_08     in_09  \
station period                                                           
12TH    Jan-Apr 2019  0.004052  0.014866  0.038180  0.068373  0.060242   
        Jan-Apr 2023  0.005339  0.024268  0.056094  0.086023  0.066539   
16TH    Jan-Apr 2019  0.005505  0.013954  0.038136  0.094354  0.085015   
        Jan-Apr 2023  0.007062  0.018760  0.040711  0.081496  0.064509   
19TH    Jan-Apr 2019  0.004275  0.016088  0.048829  0.107025  0.089471   

                         in_10     in_11     in_12     in_13     in_14  ...  \
station period                                                          ...   
12TH    Jan-Apr 2019  0.035478  0.029183  0.032836  0.033515  0.037511  ...   
        Jan-Apr 2023  0.042228  0.037291  0.039158  0.040993  0.046571  ...   
16TH    Jan-Apr 2019  0.037374  0.025771  0.026653  0.028733  0.034331  ...   
        Jan-Apr 2023  0.037292  0.031582  0.032669  0.036072  0.041893  ...   
19TH    Jan-Apr 2019  0.038018  0.025738  0.025326  0.026569  0.029387  ...   

                        out_16    out_17    out_18    out_19    out_20  \
station period                                                           
12TH    Jan-Apr 2019  0.046360  0.072190  0.061671  0.034807  0.021491   
        Jan-Apr 2023  0.064881  0.085572  0.066777  0.039255  0.025662   
16TH    Jan-Apr 2019  0.063026  0.120797  0.104764  0.057999  0.030378   
        Jan-Apr 2023  0.076021  0.108664  0.091960  0.053536  0.027405   
19TH    Jan-Apr 2019  0.051531  0.100923  0.098985  0.058757  0.030527   

                        out_21    out_22    out_23    out_00  weekend_ratio  
station period                                                               
12TH    Jan-Apr 2019  0.015686  0.012564  0.006717  0.003162       0.252671  
        Jan-Apr 2023  0.019081  0.016330  0.009502  0.003289       0.375559  
16TH    Jan-Apr 2019  0.017818  0.018180  0.010046  0.003107       0.447120  
        Jan-Apr 2023  0.018989  0.018653  0.012059  0.003409       0.589295  
19TH    Jan-Apr 2019  0.019163  0.015385  0.007394  0.002657       0.244709  

[5 rows x 41 columns]

ตรวจความถูกต้องของ feature
- สัดส่วนทั้ง 24 ชั่วโมงของแต่ละสถานีต้องรวมกันได้ 1
- ไม่มีค่าว่าง
- ทุกสถานีต้องมีข้อมูลครบทุกวันในช่วง (ไม่มีสถานีที่เปิดกลางช่วง)

In [5]:
con.register("profile_frame", profile)
largest_error = fetch_value(con, """
    WITH totals AS (
        SELECT station, period, sum(entry_share) AS entry_total, sum(exit_share) AS exit_total
        FROM profile_frame
        GROUP BY station, period
    )
    SELECT greatest(max(abs(entry_total - 1)), max(abs(exit_total - 1)))
    FROM totals
""")
print(f"largest |sum of shares - 1|: {largest_error:.2e}")
print(f"missing feature values      : {int(features.isna().sum().sum())}")

expected_days = {}
for row in daily.to_dict("records"):
    period = row["period"]
    if period not in expected_days or row["workdays"] > expected_days[period]:
        expected_days[period] = row["workdays"]

short_rows = []
for row in daily.to_dict("records"):
    if row["workdays"] < expected_days[row["period"]]:
        short_rows.append(row)
print(f"stations with fewer workdays than the period: {len(short_rows)}")

smallest = daily.sort_values("workday_entries").head(5)[["station", "period", "workday_entries"]]
smallest

largest |sum of shares - 1|: 4.44e-16
missing feature values      : 0
stations with fewer workdays than the period: 0


,station,period,workday_entries
65,PCTR,Jan-Apr 2023,538.121951
59,NCON,Jan-Apr 2023,599.621951
61,OAKL,Jan-Apr 2023,653.548780
53,MLPT,Jan-Apr 2023,921.695122
95,WDUB,Jan-Apr 2023,1072.097561


สถานีที่เล็กที่สุดยังมีคนเข้าวันละ ~500 คน รวม ~80 วันทำงาน สัดส่วนรายชั่วโมงจึงคำนวณจากข้อมูลมากพอ ไม่แกว่งเพราะสุ่ม

## 2. เลือกจำนวนกลุ่ม (k)

สำหรับ k = 2 ถึง 8 วัด 3 อย่าง
- **silhouette**: แต่ละสถานีใกล้กลุ่มตัวเองมากกว่ากลุ่มอื่นแค่ไหน (-1 ถึง 1 ยิ่งสูงยิ่งแยกชัด)
- **ความเสถียร**: รัน KMeans 50 seed แล้วเทียบผลทุกคู่ด้วย ARI (1 = แบ่งเหมือนกันทุกครั้ง) ถ้าต่ำแปลว่ากลุ่มขึ้นกับการสุ่ม เชื่อถือไม่ได้
- **ตรงกับ Ward แค่ไหน**: ARI ระหว่าง KMeans กับ Hierarchical clustering (Ward) ถ้าสองวิธีให้ผลคล้ายกัน แปลว่ากลุ่มมีอยู่จริงในข้อมูล ไม่ได้เกิดจากวิธีที่เลือก

In [6]:
scaled = StandardScaler().fit_transform(features[feature_columns])


def make_kmeans(k, seed, restarts):
    # each fit tries `restarts` starting points and keeps the one with the lowest inertia
    model = KMeans(n_clusters=k, random_state=seed)
    model.set_params(n_init=restarts)
    return model


def stability(k, seeds=50):
    all_labels = []
    for seed in range(seeds):
        model = make_kmeans(k, seed, restarts=10)
        all_labels.append(model.fit_predict(scaled))

    scores = []
    for first in range(seeds):
        for second in range(first + 1, seeds):
            scores.append(adjusted_rand_score(all_labels[first], all_labels[second]))
    return float(np.mean(scores)), float(np.min(scores))


def cluster_sizes(labels):
    counts = np.bincount(labels)
    sizes = []
    for count in sorted(counts, reverse=True):
        sizes.append(str(count))
    return " / ".join(sizes)


k_rows = []
with stage("09.choose_k"):
    for k in range(2, 9):
        model = make_kmeans(k, seed=42, restarts=50)
        labels = model.fit_predict(scaled)
        ward_labels = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(scaled)
        mean_ari, min_ari = stability(k)
        k_rows.append({
            "k": k,
            "silhouette": round(float(silhouette_score(scaled, labels)), 3),
            "stability_mean_ari": round(mean_ari, 3),
            "stability_min_ari": round(min_ari, 3),
            "ari_vs_ward": round(float(adjusted_rand_score(labels, ward_labels)), 3),
            "cluster_sizes": cluster_sizes(labels),
        })

k_table = pd.DataFrame(k_rows).set_index("k")
k_table

[perf] 09.choose_k: 6.05s | 7.66 cores | peak RAM 0.392 GB | CPU 39.0%


,silhouette,stability_mean_ari,stability_min_ari,ari_vs_ward,cluster_sizes
k,,,,,
2,0.490,1.000,1.000,1.000,77 / 21
3,0.246,0.998,0.972,0.380,41 / 37 / 20
4,0.262,0.992,0.965,0.650,41 / 37 / 11 / 9
5,0.261,0.840,0.437,0.698,40 / 37 / 10 / 7 / 4
6,0.268,0.647,0.333,0.555,39 / 37 / 12 / 5 / 4 / 1
7,0.243,0.603,0.311,0.475,37 / 20 / 19 / 9 / 8 / 4 / 1
8,0.211,0.649,0.336,0.650,25 / 18 / 18 / 15 / 9 / 8 / 4 / 1


In [7]:
fig = go.Figure()
measures = [
    ("silhouette", "silhouette", viz.SERIES[0]),
    ("stability_mean_ari", "stability (mean ARI over 50 seeds)", viz.SERIES[1]),
    ("ari_vs_ward", "agreement with Ward (ARI)", viz.SERIES[2]),
]
for column, label, color in measures:
    fig.add_trace(go.Scatter(
        x=k_table.index,
        y=k_table[column],
        mode="lines+markers",
        name=label,
        line=dict(color=color, width=2),
        marker=dict(size=8),
        hovertemplate="k=%{x}: %{y:.3f}<extra>" + label + "</extra>",
    ))
viz.vline(fig, 4, "chosen k = 4", y=1.02)
fig.update_xaxes(title_text="number of clusters (k)", dtick=1)
fig.update_yaxes(title_text="score (0 to 1)", range=[0, 1.05])

title = "Four station types are the most detailed split that stays stable"
subtitle = "k = 2 separates only origins from destinations; from k = 5 the clusters change with the random seed"
viz.titled(fig, title, subtitle)
viz.save(fig, "09_choose_k")
fig

เลือก k = 4
- k = 2 silhouette สูงสุด แต่แบ่งได้แค่ "ต้นทาง" กับ "ปลายทาง" ซึ่งรู้อยู่แล้ว ไม่ช่วยตอบว่าบทบาทเปลี่ยนอย่างไร
- k = 4 ยังเสถียรมาก (ARI เฉลี่ย ~0.99 ทุก seed ได้ผลเกือบเหมือนกัน) silhouette สูงกว่า k = 3 และตรงกับ Ward มากกว่า k = 3
- ตั้งแต่ k = 5 ความเสถียรลดลงชัด (ARI ต่ำสุดเหลือ ~0.4) กลุ่มที่ได้ขึ้นกับการสุ่ม จึงไม่ควรใช้

silhouette ของ k = 4 อยู่ที่ ~0.26 แปลว่ากลุ่มแยกกันพอใช้ได้ แต่ไม่ได้ขาดจากกันชัด สถานีบางแห่งอยู่ระหว่างสองกลุ่ม (ดูข้อ 4)

In [8]:
CHOSEN_K = 4

with stage("09.final_kmeans"):
    final_model = make_kmeans(CHOSEN_K, seed=42, restarts=50)
    raw_labels = final_model.fit_predict(scaled)

print(f"inertia: {final_model.inertia_:.1f}")
print(f"silhouette: {silhouette_score(scaled, raw_labels):.3f}")

[perf] 09.final_kmeans: 0.02s | 12.59 cores | peak RAM 0.396 GB
inertia: 1430.0
silhouette: 0.262


## 3. ตั้งชื่อกลุ่มจาก centroid

เลขกลุ่มจาก KMeans เป็นเลขสุ่ม จึงตั้งชื่อด้วยกฎที่อ่านจากรูปทรงจริง
- ช่วงเช้า (6:00-9:59) ถ้าคนเข้าสถานีมากกว่าคนออก แปลว่าเป็น **ต้นทาง** (คนออกจากบ้าน) ถ้าคนออกมากกว่า แปลว่าเป็น **ปลายทาง** (คนมาทำงาน)
- ในกลุ่มต้นทาง กลุ่มที่สัดส่วนคนเข้าช่วงเช้าสูงกว่าคือ "ต้นทางคนทำงาน (peak แหลม)" อีกกลุ่มคือ "ต้นทางแบบผสม (peak เตี้ยกว่า)"
- ในกลุ่มปลายทาง กลุ่มที่สัดส่วนคนออกช่วงเช้าสูงกว่าคือ "ย่านทำงาน" อีกกลุ่มคือ "ปลายทางทั้งวัน"

ถ้าผลไม่ได้เป็นต้นทาง 2 กลุ่มและปลายทาง 2 กลุ่ม โค้ดจะหยุดทันที เพื่อไม่ให้ตั้งชื่อผิด

In [9]:
AM_HOURS = [6, 7, 8, 9]

centroids = pd.DataFrame(features[feature_columns].groupby(raw_labels).mean())

centroid_rows = []
for cluster_id, row in centroids.iterrows():
    am_in = 0.0
    am_out = 0.0
    for hour in AM_HOURS:
        am_in = am_in + row[f"in_{hour:02d}"]
        am_out = am_out + row[f"out_{hour:02d}"]
    centroid_rows.append({"cluster_id": cluster_id, "am_in": am_in, "am_out": am_out})

origins = []
destinations = []
for row in centroid_rows:
    if row["am_in"] > row["am_out"]:
        origins.append(row)
    else:
        destinations.append(row)

if len(origins) != 2 or len(destinations) != 2:
    raise ValueError(f"expected 2 origin and 2 destination clusters, got {len(origins)} and {len(destinations)}")


def by_am_in(row):
    return row["am_in"]


def by_am_out(row):
    return row["am_out"]


origins.sort(key=by_am_in, reverse=True)
destinations.sort(key=by_am_out, reverse=True)

COMMUTER = "commuter origin"
MIXED = "mixed origin"
DOWNTOWN = "downtown job centre"
ALL_DAY = "all-day destination"
CLUSTER_ORDER = [COMMUTER, MIXED, DOWNTOWN, ALL_DAY]
CLUSTER_COLOR = {
    COMMUTER: viz.SERIES[0],
    MIXED: viz.SERIES[2],
    DOWNTOWN: viz.SERIES[1],
    ALL_DAY: viz.SERIES[3],
}

cluster_name = {}
cluster_name[origins[0]["cluster_id"]] = COMMUTER
cluster_name[origins[1]["cluster_id"]] = MIXED
cluster_name[destinations[0]["cluster_id"]] = DOWNTOWN
cluster_name[destinations[1]["cluster_id"]] = ALL_DAY

names = []
for label in raw_labels:
    names.append(cluster_name[label])

assignments = features.index.to_frame(index=False)
assignments["cluster"] = names
assignments["year"] = assignments["period"].str[-4:]
assignments.head()

,station,period,cluster,year
0,12TH,Jan-Apr 2019,downtown job centre,2019
1,12TH,Jan-Apr 2023,downtown job centre,2023
2,16TH,Jan-Apr 2019,downtown job centre,2019
3,16TH,Jan-Apr 2023,all-day destination,2023
4,19TH,Jan-Apr 2019,downtown job centre,2019


สรุปแต่ละกลุ่ม: จำนวนสถานีในแต่ละปี และรูปทรงเฉลี่ย (ค่าเป็น % ของผู้โดยสารทั้งวัน)

In [10]:
def hour_sum(frame, prefix, hours):
    total = pd.Series(0.0, index=frame.index)
    for hour in hours:
        total = total + frame[f"{prefix}_{hour:02d}"]
    return total


PM_HOURS = [15, 16, 17, 18]

summary_input = features.copy()
summary_input["cluster"] = names
summary_input["am_in_pct"] = 100 * hour_sum(summary_input, "in", AM_HOURS)
summary_input["am_out_pct"] = 100 * hour_sum(summary_input, "out", AM_HOURS)
summary_input["pm_in_pct"] = 100 * hour_sum(summary_input, "in", PM_HOURS)
summary_input["pm_out_pct"] = 100 * hour_sum(summary_input, "out", PM_HOURS)

cluster_summary = summary_input.groupby("cluster").agg(
    am_in_pct=("am_in_pct", "mean"),
    am_out_pct=("am_out_pct", "mean"),
    pm_in_pct=("pm_in_pct", "mean"),
    pm_out_pct=("pm_out_pct", "mean"),
    weekend_ratio=("weekend_ratio", "mean"),
).round(2)

counts = assignments.groupby(["cluster", "year"]).size().unstack(fill_value=0)
cluster_summary = cluster_summary.join(counts).loc[CLUSTER_ORDER]
cluster_summary

,am_in_pct,am_out_pct,pm_in_pct,pm_out_pct,weekend_ratio,2019,2023
cluster,,,,,,,
commuter origin,60.36,9.44,13.78,56.58,0.32,30,7
mixed origin,45.22,16.66,22.78,47.90,0.46,8,33
downtown job centre,14.68,52.67,53.72,20.00,0.32,6,5
all-day destination,12.96,32.59,38.87,27.99,0.64,4,5


In [11]:
members = []
for cluster in CLUSTER_ORDER:
    for year in ["2019", "2023"]:
        rows = assignments[(assignments["cluster"] == cluster) & (assignments["year"] == year)]
        stations = sorted(rows["station"])
        members.append({"cluster": cluster, "year": year, "stations": ", ".join(stations)})

pd.set_option("display.max_colwidth", 200)
pd.DataFrame(members)

,cluster,year,stations
0,commuter origin,2019,"ANTC, ASHB, BAYF, CAST, COLM, CONC, DELN, DUBL, FRMT, GLEN, LAFY, MLBR, NBRK, NCON, ORIN, PCTR, PHIL, PITT, PLZA, RICH, ROCK, SANL, SBRN, SHAY, SSAN, UCTY, WARM, WCRK, WDUB, WOAK"
1,commuter origin,2023,"ANTC, CAST, DUBL, NCON, PCTR, PHIL, PITT"
2,mixed origin,2019,"24TH, BALB, COLS, DALY, FTVL, HAYW, LAKE, MCAR"
3,mixed origin,2023,"19TH, 24TH, ASHB, BALB, BAYF, BERY, COLM, COLS, CONC, DALY, DELN, FRMT, FTVL, GLEN, HAYW, LAFY, LAKE, MCAR, MLBR, MLPT, NBRK, ORIN, PLZA, RICH, ROCK, SANL, SBRN, SHAY, SSAN, UCTY, WCRK, WDUB, WOAK"
4,downtown job centre,2019,"12TH, 16TH, 19TH, CIVC, EMBR, MONT"
5,downtown job centre,2023,"12TH, CIVC, EMBR, MONT, WARM"
6,all-day destination,2019,"DBRK, OAKL, POWL, SFIA"
7,all-day destination,2023,"16TH, DBRK, OAKL, POWL, SFIA"


In [12]:
fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    subplot_titles=["entries (people entering the station)", "exits (people leaving the station)"],
    horizontal_spacing=0.06,
)
hour_labels = []
for hour in FEATURE_HOURS:
    hour_labels.append(f"{hour:02d}")

for cluster in CLUSTER_ORDER:
    rows = summary_input[summary_input["cluster"] == cluster]
    entry_values = []
    exit_values = []
    for hour in FEATURE_HOURS:
        entry_values.append(100 * rows[f"in_{hour:02d}"].mean())
        exit_values.append(100 * rows[f"out_{hour:02d}"].mean())

    fig.add_trace(go.Scatter(
        x=hour_labels, y=entry_values, mode="lines", name=cluster, legendgroup=cluster,
        line=dict(color=CLUSTER_COLOR[cluster], width=2),
        hovertemplate="%{x}:00  %{y:.1f}% of daily entries<extra>" + cluster + "</extra>",
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=hour_labels, y=exit_values, mode="lines", name=cluster, legendgroup=cluster, showlegend=False,
        line=dict(color=CLUSTER_COLOR[cluster], width=2),
        hovertemplate="%{x}:00  %{y:.1f}% of daily exits<extra>" + cluster + "</extra>",
    ), row=1, col=2)

fig.update_xaxes(
    title_text="hour",
    type="category",
    categoryorder="array",
    categoryarray=hour_labels,
    tickvals=["06", "09", "12", "15", "18", "21", "00"],
)
fig.update_yaxes(title_text="% of the day", row=1, col=1)
fig.update_layout(hovermode="closest", margin=dict(t=150), legend=dict(y=1.13))
fig.update_annotations(font=dict(size=12, color=viz.INK2))

title = "Four station types: two kinds of origin, two kinds of destination"
subtitle = "Average hourly share of a workday's entries and exits, station-periods in each cluster (2019 and 2023 together)"
viz.titled(fig, title, subtitle)
viz.save(fig, "09_cluster_profiles", height=540)
fig

## 4. แผนที่สถานี (PCA) และการย้ายกลุ่ม

ย่อ feature 41 มิติเหลือ 2 มิติด้วย PCA เพื่อวาดภาพ (ใช้แค่วาดภาพ ไม่ได้ใช้จัดกลุ่ม)
ลูกศรลากจากตำแหน่งของสถานีปี 2019 ไปปี 2023

In [13]:
pca = PCA(n_components=2, random_state=42)
coordinates = pca.fit_transform(scaled)
explained = pca.explained_variance_ratio_
print(f"PC1 explains {100 * explained[0]:.1f}% and PC2 {100 * explained[1]:.1f}% of the variance")

assignments["pc1"] = coordinates[:, 0]
assignments["pc2"] = coordinates[:, 1]

loadings = pd.DataFrame(np.asarray(pca.components_).T, index=feature_columns, columns=["pc1", "pc2"])
loadings["abs_pc1"] = np.abs(loadings["pc1"])
loadings.sort_values("abs_pc1", ascending=False).head(8).round(3)

PC1 explains 59.6% and PC2 17.7% of the variance


,pc1,pc2,abs_pc1
out_10,0.194,0.023,0.194
out_17,-0.193,0.014,0.193
in_08,-0.192,-0.020,0.192
in_16,0.192,-0.048,0.192
in_19,0.187,-0.051,0.187
out_09,0.186,-0.112,0.186
in_15,0.186,0.065,0.186
out_18,-0.185,-0.040,0.185


In [14]:
wide = assignments.pivot(index="station", columns="year", values=["cluster", "pc1", "pc2"])
moves = pd.DataFrame({
    "cluster_2019": wide[("cluster", "2019")],
    "cluster_2023": wide[("cluster", "2023")],
    "pc1_2019": wide[("pc1", "2019")],
    "pc2_2019": wide[("pc2", "2019")],
    "pc1_2023": wide[("pc1", "2023")],
    "pc2_2023": wide[("pc2", "2023")],
})
both_years = moves.dropna().copy()
both_years["switched"] = both_years["cluster_2019"] != both_years["cluster_2023"]
both_years.index.name = "station"
station_records = both_years.reset_index().to_dict("records")

switched_count = 0
for row in station_records:
    if row["switched"]:
        switched_count = switched_count + 1

print(f"stations in both years: {len(both_years)}")
print(f"stations that switched cluster: {switched_count}")

LABELLED = ["WARM", "16TH", "19TH", "COLM", "ORIN", "EMBR", "SFIA", "PITT", "ANTC", "POWL"]

fig = go.Figure()
for row in station_records:
    if row["switched"]:
        arrow_color = viz.INK2
        arrow_width = 1.2
    else:
        arrow_color = viz.CONTEXT
        arrow_width = 0.8
    fig.add_annotation(
        x=row["pc1_2023"], y=row["pc2_2023"], ax=row["pc1_2019"], ay=row["pc2_2019"],
        xref="x", yref="y", axref="x", ayref="y",
        showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=arrow_width, arrowcolor=arrow_color, text="",
    )

points_2019 = assignments[assignments["year"] == "2019"]
fig.add_trace(go.Scatter(
    x=points_2019["pc1"], y=points_2019["pc2"], mode="markers", name="2019 position",
    marker=dict(size=8, color="white", line=dict(color=viz.MUTED, width=1.5)),
    text=points_2019["station"], customdata=points_2019["cluster"],
    hovertemplate="%{text} 2019<br>%{customdata}<extra></extra>",
))

for cluster in CLUSTER_ORDER:
    rows = assignments[(assignments["year"] == "2023") & (assignments["cluster"] == cluster)]
    labels = []
    for station in rows["station"]:
        if station in LABELLED:
            labels.append(station)
        else:
            labels.append("")
    fig.add_trace(go.Scatter(
        x=rows["pc1"], y=rows["pc2"], mode="markers+text", name=f"2023: {cluster}",
        marker=dict(size=10, color=CLUSTER_COLOR[cluster], line=dict(color="white", width=1)),
        text=labels, textposition="top center", textfont=dict(size=11, color=viz.INK),
        customdata=rows["station"],
        hovertemplate="%{customdata} 2023<extra>" + cluster + "</extra>",
    ))

fig.update_layout(hovermode="closest", legend=dict(y=1.0))
fig.update_xaxes(title_text=f"PC1 ({100 * explained[0]:.0f}% of variance): origin (left) to destination (right)", showgrid=True, gridcolor=viz.GRID)
fig.update_yaxes(title_text=f"PC2 ({100 * explained[1]:.0f}%)")

title = f"{switched_count} of {len(both_years)} stations changed type between 2019 and 2023"
subtitle = "Each arrow runs from a station's 2019 position to its 2023 position; dark arrows mark a change of cluster"
viz.titled(fig, title, subtitle)
viz.save(fig, "09_station_map", height=640)
fig

stations in both years: 48
stations that switched cluster: 25


ตารางการย้ายกลุ่ม: แถวคือกลุ่มปี 2019 คอลัมน์คือกลุ่มปี 2023

In [15]:
transition = pd.crosstab(both_years["cluster_2019"], both_years["cluster_2023"])
transition = transition.reindex(index=CLUSTER_ORDER, columns=CLUSTER_ORDER, fill_value=0)
transition

cluster_2023,commuter origin,mixed origin,downtown job centre,all-day destination
cluster_2019,,,,
commuter origin,7,22,1,0
mixed origin,0,8,0,0
downtown job centre,0,1,4,1
all-day destination,0,0,0,4


In [16]:
am_share = summary_input["am_in_pct"].unstack("period")
am_exit_share = summary_input["am_out_pct"].unstack("period")

switch_rows = []
for row in station_records:
    if not row["switched"]:
        continue
    station = row["station"]
    switch_rows.append({
        "station": station,
        "cluster_2019": row["cluster_2019"],
        "cluster_2023": row["cluster_2023"],
        "am_entries_pct_2019": round(am_share.loc[station, BEFORE], 1),
        "am_entries_pct_2023": round(am_share.loc[station, AFTER], 1),
        "am_exits_pct_2019": round(am_exit_share.loc[station, BEFORE], 1),
        "am_exits_pct_2023": round(am_exit_share.loc[station, AFTER], 1),
    })

switched_table = pd.DataFrame(switch_rows).sort_values(["cluster_2019", "cluster_2023", "station"])
switched_table

,station,cluster_2019,cluster_2023,am_entries_pct_2019,am_entries_pct_2023,am_exits_pct_2019,am_exits_pct_2023
21,WARM,commuter origin,downtown job centre,51.5,31.0,18.1,36.0
2,ASHB,commuter origin,mixed origin,52.7,43.1,11.9,14.2
3,BAYF,commuter origin,mixed origin,61.5,51.5,7.3,10.2
4,COLM,commuter origin,mixed origin,67.7,53.5,6.1,11.7
5,CONC,commuter origin,mixed origin,60.7,50.2,10.0,13.1
6,DELN,commuter origin,mixed origin,62.5,55.8,5.9,8.0
7,FRMT,commuter origin,mixed origin,58.1,45.0,12.2,18.4
8,GLEN,commuter origin,mixed origin,51.9,41.8,14.8,19.1
9,LAFY,commuter origin,mixed origin,62.4,53.6,7.2,11.7
10,MLBR,commuter origin,mixed origin,51.1,36.3,18.8,21.5


## 5. peak เช้าของสถานีต้นทางเตี้ยลงเท่าไร

ตัวเลขที่อ่านง่ายที่สุดของการย้ายกลุ่มคือ สัดส่วนคนเข้าสถานีช่วง 6:00-9:59 ต่อทั้งวัน
ถ้าค่านี้ลดลง แปลว่าผู้โดยสารกระจายไปช่วงอื่นของวันมากขึ้น ไม่ได้ออกจากบ้านพร้อมกันตอนเช้าเหมือนก่อน
ดูเฉพาะสถานีที่ปี 2019 เป็นต้นทาง (2 กลุ่มแรก)

In [17]:
origin_rows = []
for row in station_records:
    station = row["station"]
    if row["cluster_2019"] in [COMMUTER, MIXED]:
        origin_rows.append({
            "station": station,
            "cluster_2019": row["cluster_2019"],
            "cluster_2023": row["cluster_2023"],
            "am_2019": am_share.loc[station, BEFORE],
            "am_2023": am_share.loc[station, AFTER],
        })

origin_change = pd.DataFrame(origin_rows)
origin_change["change_points"] = origin_change["am_2023"] - origin_change["am_2019"]
origin_change = origin_change.sort_values("am_2019")

fell = int((origin_change["change_points"] < 0).sum())
print(f"origin stations: {len(origin_change)}, AM share fell at {fell}")
print(f"median change: {origin_change['change_points'].median():.1f} points")
origin_change.groupby("cluster_2019")[["am_2019", "am_2023", "change_points"]].median().round(1)

origin stations: 38, AM share fell at 37
median change: -9.0 points


,am_2019,am_2023,change_points
cluster_2019,,,
commuter origin,59.9,50.8,-9.8
mixed origin,48.0,39.6,-6.7


In [18]:
fig = go.Figure()
for row in origin_change.to_dict("records"):
    fig.add_trace(go.Scatter(
        x=[row["am_2019"], row["am_2023"]], y=[row["station"], row["station"]], mode="lines",
        line=dict(color=viz.CONTEXT, width=2), showlegend=False, hoverinfo="skip",
    ))

fig.add_trace(go.Scatter(
    x=origin_change["am_2019"], y=origin_change["station"], mode="markers", name="Jan-Apr 2019",
    marker=dict(size=9, color=viz.MUTED),
    customdata=origin_change["cluster_2019"],
    hovertemplate="%{y} 2019: %{x:.1f}%<br>%{customdata}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=origin_change["am_2023"], y=origin_change["station"], mode="markers", name="Jan-Apr 2023",
    marker=dict(size=9, color=viz.ACCENT),
    customdata=origin_change["cluster_2023"],
    hovertemplate="%{y} 2023: %{x:.1f}%<br>%{customdata}<extra></extra>",
))

fig.update_layout(hovermode="closest", margin=dict(l=70, t=140))
fig.update_xaxes(title_text="% of a workday's entries made 6:00-9:59", showgrid=True, gridcolor=viz.GRID, ticksuffix="%")
fig.update_yaxes(showgrid=False)

title = f"The morning rush flattened at {fell} of {len(origin_change)} origin stations"
subtitle = "Share of workday entries made 6:00-9:59, stations that were origins in 2019, sorted by the 2019 share"
viz.titled(fig, title, subtitle)
viz.save(fig, "09_am_share_change", height=900)
fig

## 6. แต่ละกลุ่มฟื้นตัวเท่าไร

ใช้กลุ่มของปี 2019 (บทบาทเดิม) แล้วเทียบ entries เฉลี่ยต่อวันทำงานปี 2023 กับปี 2019 (ไม่รวม MLPT และ BERY ที่ไม่มีข้อมูลปี 2019)

In [19]:
volume = daily.pivot(index="station", columns="period", values="workday_entries")
volume = volume.loc[both_years.index]

recovery_rows = []
for cluster in CLUSTER_ORDER:
    stations = both_years[both_years["cluster_2019"] == cluster].index
    before_total = volume.loc[stations, BEFORE].sum()
    after_total = volume.loc[stations, AFTER].sum()
    recovery_rows.append({
        "cluster_2019": cluster,
        "stations": len(stations),
        "entries_2019": round(before_total),
        "entries_2023": round(after_total),
        "recovery_pct": round(100 * after_total / before_total, 1),
    })

system_before = volume[BEFORE].sum()
system_after = volume[AFTER].sum()
system_recovery = 100 * system_after / system_before
recovery = pd.DataFrame(recovery_rows).set_index("cluster_2019")
print(f"all {len(volume)} stations: {system_recovery:.1f}% of 2019")
recovery

all 48 stations: 36.6% of 2019


,stations,entries_2019,entries_2023,recovery_pct
cluster_2019,,,,
commuter origin,30,145658,54390,37.3
mixed origin,8,67454,27274,40.4
downtown job centre,6,149955,47661,31.8
all-day destination,4,46789,20495,43.8


In [20]:
fig = go.Figure(go.Bar(
    x=recovery["recovery_pct"],
    y=recovery.index,
    orientation="h",
    marker_color=viz.ACCENT,
    text=recovery["recovery_pct"].map("{:.0f}%".format),
    textposition="outside",
    hovertemplate="%{y}: %{x:.1f}% of 2019<extra></extra>",
))
viz.vline(fig, system_recovery, f"all stations {system_recovery:.0f}%", y=1.02)
fig.update_layout(hovermode="closest", showlegend=False, margin=dict(l=170))
fig.update_xaxes(title_text="workday entries Jan-Apr 2023 as % of Jan-Apr 2019", range=[0, 55], ticksuffix="%", showgrid=True, gridcolor=viz.GRID)
fig.update_yaxes(showgrid=False, autorange="reversed")

title = "Downtown job centres recovered least, all-day destinations most"
subtitle = "Stations grouped by their 2019 type"
viz.titled(fig, title, subtitle)
viz.save(fig, "09_recovery_by_type", height=420)
fig

อัตราส่วนผู้โดยสารวันหยุด/เสาร์-อาทิตย์ต่อวันทำงาน (ค่าเฉลี่ยของสถานีในแต่ละกลุ่ม ใช้กลุ่มของปีนั้นๆ)

In [21]:
weekend = daily.pivot(index="station", columns="period", values="weekend_ratio").loc[both_years.index]
rose = int((weekend[AFTER] > weekend[BEFORE]).sum())
print(f"weekend ratio rose at {rose} of {len(weekend)} stations")
print(f"median: {weekend[BEFORE].median():.2f} in 2019, {weekend[AFTER].median():.2f} in 2023")

weekend_by_type = summary_input.reset_index().pivot_table(index="cluster", columns="period", values="weekend_ratio", aggfunc="mean")
weekend_by_type.loc[CLUSTER_ORDER].round(2)

weekend ratio rose at 48 of 48 stations
median: 0.33 in 2019, 0.43 in 2023


period,Jan-Apr 2019,Jan-Apr 2023
cluster,,
commuter origin,0.31,0.35
mixed origin,0.39,0.47
downtown job centre,0.27,0.37
all-day destination,0.59,0.68


## 7. ตรวจว่าผลไม่ได้ขึ้นกับวิธีที่เลือก

1. เทียบกับ Ward ที่ k = 4: สถานีไหนที่สองวิธีจัดไม่ตรงกัน
2. เปลี่ยน k เป็น 3 และ 5 แล้วดูว่าสถานีต้นทางที่ "เปลี่ยนกลุ่ม" ยังเป็นชุดเดิมไหม

In [22]:
ward_labels = AgglomerativeClustering(n_clusters=CHOSEN_K, linkage="ward").fit_predict(scaled)
assignments["ward_group"] = ward_labels
ward_table = pd.crosstab(assignments["cluster"], assignments["ward_group"]).reindex(CLUSTER_ORDER)
print(f"ARI KMeans vs Ward: {adjusted_rand_score(raw_labels, ward_labels):.3f}")
ward_table

ARI KMeans vs Ward: 0.650


ward_group,0,1,2,3
cluster,,,,
commuter origin,0,7,0,30
mixed origin,1,38,0,2
downtown job centre,11,0,0,0
all-day destination,5,0,4,0


In [23]:
def switched_stations(labels):
    frame = features.index.to_frame(index=False)
    frame["label"] = labels
    pivot = frame.pivot(index="station", columns="period", values="label").dropna()
    result = set()
    for station, row in pivot.iterrows():
        if row[BEFORE] != row[AFTER]:
            result.add(station)
    return result


base_switched = switched_stations(raw_labels)
robust_rows = []
for k in [3, 4, 5]:
    labels = make_kmeans(k, seed=42, restarts=50).fit_predict(scaled)
    switched = switched_stations(labels)
    overlap = len(switched & base_switched)
    union = len(switched | base_switched)
    robust_rows.append({
        "k": k,
        "stations_switched": len(switched),
        "same_as_k4": overlap,
        "jaccard_vs_k4": round(overlap / union, 2),
        "only_in_this_k": ", ".join(sorted(switched - base_switched)),
        "missing_vs_k4": ", ".join(sorted(base_switched - switched)),
    })

pd.DataFrame(robust_rows).set_index("k")

,stations_switched,same_as_k4,jaccard_vs_k4,only_in_this_k,missing_vs_k4
k,,,,,
3,24,24,0.96,,16TH
4,25,25,1.00,,
5,24,23,0.88,POWL,"16TH, 19TH"


In [24]:
output = assignments[["station", "period", "cluster", "pc1", "pc2"]].copy()
output = output.merge(daily[["station", "period", "workday_entries", "weekend_ratio"]], on=["station", "period"])
peak_shares = summary_input[["am_in_pct", "am_out_pct"]].reset_index()
peak_shares.columns = ["station", "period", "am_entries_pct", "am_exits_pct"]
output = output.merge(peak_shares, on=["station", "period"])
output.to_csv(TAB / "m2_station_clusters.csv", index=False)
print(f"saved {len(output)} rows to m2_station_clusters.csv")

saved 98 rows to m2_station_clusters.csv


## สรุป: Insight → Pain point

pain point ของ BART ในข้อนี้คือ ตารางเดินรถ ความยาวขบวน และพนักงานสถานียังวางตามรูปแบบก่อน COVID ต้องรู้ว่าสถานีไหนเปลี่ยนบทบาทไปแล้วบ้าง

| # | Insight (ตัวเลข) | Pain point | ตอบโจทย์แค่ไหน | ข้อเสนอ |
|---|---|---|---|---|
| 1 | สถานีแบ่งได้ 4 แบบที่เสถียร (ARI ข้าม 50 seed ~0.99) ต้นทางคนทำงาน, ต้นทางแบบผสม, ย่านทำงาน และปลายทางทั้งวัน ปี 2019 มี 30 / 8 / 6 / 4 สถานี | แผนเดียวใช้กับทุกสถานี | ตรงจุด | วางแม่แบบตารางพนักงานและการเปิดประตูกั้นแยกตาม 4 แบบ แทนการใช้แบบเดียวทั้งระบบ |
| 2 | 22 จาก 30 สถานีต้นทางคนทำงานย้ายไปเป็นต้นทางแบบผสม สัดส่วนคนเข้าสถานีช่วง 6:00-9:59 ลดลงที่ 37 จาก 38 สถานีต้นทาง ค่ามัธยฐานของกลุ่มต้นทางคนทำงานจาก 59.9% เหลือ 50.8% | ความจุช่วง peak เช้า | ตรงจุด | ลดสัดส่วนขบวนยาวช่วงเช้าลง แล้วย้ายความจุบางส่วนไปช่วงสายถึงบ่าย สอดคล้องกับ Q2 ที่ peak เตี้ยและกว้างขึ้น |
| 3 | เหลือต้นทางคนทำงานแบบเดิมแค่ 7 สถานี ANTC, PCTR, PITT, NCON, PHIL (สาย Antioch) และ CAST, DUBL (สาย Dublin) สัดส่วนเข้าช่วงเช้ายัง 52-70% | ควรเก็บความจุ peak ไว้ที่ไหน | บางส่วน บอกได้ระดับสถานี แต่โหลดบนขบวนต้องดูจาก Q3 | ถ้าต้องลดขบวนยาวช่วงเช้า ให้เก็บไว้สองสายนี้ก่อน |
| 4 | ย่านทำงานของปี 2019 (12TH, 16TH, 19TH, CIVC, EMBR, MONT) ฟื้นน้อยที่สุด 31.8% ของปี 2019 ต่ำกว่าทั้งระบบ (36.6%) ส่วนปลายทางทั้งวัน (SFIA, OAKL, POWL, DBRK) ฟื้นมากที่สุด 43.8% | สถานีไหนควรลดหรือคงบริการ | ตรงจุด สอดคล้องกับ Q1 ที่ทริปเข้า-ออกตัวเมือง SF คิดเป็น 72% ของทริปที่หายไป | สถานีย่านทำงานยังคงรูปทรงเดิม แค่คนน้อยลง ให้ลดความยาวขบวนหรือพนักงาน แต่ไม่ต้องเปลี่ยนช่วงเวลา |
| 5 | อัตราส่วนวันหยุด/เสาร์-อาทิตย์ต่อวันทำงานเพิ่มขึ้นทั้ง 48 สถานี ค่ามัธยฐานจาก 0.33 เป็น 0.43 ปลายทางทั้งวันสูงสุดที่ 0.68 | บริการวันหยุด | ตรงจุด | ไม่ควรลดบริการวันหยุดตามสัดส่วนของวันทำงาน เพราะวันหยุดสำคัญขึ้นในทุกสถานี (สอดคล้องกับ Q1) |
| 6 | WARM เปลี่ยนจากต้นทางเป็นย่านทำงาน คนเข้าช่วงเช้าจาก 51.5% เหลือ 31.0% และคนออกช่วงเช้าจาก 18.1% เป็น 36.0% เพราะผู้โดยสารต้นทางย้ายไป MLPT/BERY (Q4) ส่วน 16TH เปลี่ยนเป็นปลายทางทั้งวัน | พนักงานและการจัดการประตูรายสถานี | บางส่วน เป็นรายสถานี และ 19TH อยู่ก้ำกึ่งระหว่างกลุ่ม (เปลี่ยนกลุ่มที่ k = 4 แต่ไม่เปลี่ยนที่ k = 5) | ทบทวนพนักงานช่วงเช้าของ WARM ใหม่ในฐานะสถานีปลายทาง และประสานรถ feeder ไปย่านงานรอบสถานี |

ทรัพยากรที่ใช้: สรุปรายชั่วโมงด้วย DuckDB ไม่ถึง 1 วินาที, เลือก k (7 ค่า × 50 seed) ~5 วินาที, ทั้ง notebook ~15 วินาที บน CPU

ข้อจำกัด
- silhouette ~0.26 แปลว่ารูปแบบสถานีเป็นช่วงต่อเนื่อง ไม่ได้แยกขาดเป็นกล่อง ต้นทาง 2 แบบต่างกันที่ "ความแหลมของ peak เช้า" ควรดูตัวเลขสัดส่วนช่วงเช้า (ข้อ 5 ใน notebook) ประกอบกับชื่อกลุ่ม
- Ward ตรงกับ KMeans แค่ปานกลาง (ARI 0.65) เพราะแบ่งปลายทาง 2 แบบต่างกัน แต่ชุดสถานีที่เปลี่ยนกลุ่มยังเกือบเหมือนเดิมเมื่อเปลี่ยน k เป็น 3 หรือ 5 (Jaccard 0.96 และ 0.88)
- ใช้แค่ ม.ค.-เม.ย. ของสองปี และดู entries/exits ระดับสถานี ไม่ได้ดูว่าผู้โดยสารไปที่ไหน (ดู Q3) หรือโหลดบนขบวน

ต่อไป: 10_m3_backtest_events